## Phase 6 SEC Ablation (Strict Local Runbook)

Run top-to-bottom:
1. Local project setup
2. Install dependencies (if needed)
3. Build strict SEC FinBERT embeddings (10-Q/10-K MD&A -> daily panel)
4. Run Phase 6 SEC ablation using strict embedding features
5. Inspect artifacts

In [6]:
from pathlib import Path
import os
import sys

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'phase6_sec_ppo.py').exists():
    PROJECT_DIR = Path('/Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project ')

os.chdir(PROJECT_DIR)
print('cwd:', os.getcwd())
print('python:', sys.executable)
print('phase6 script exists:', Path('phase6_sec_ppo.py').exists())
print('train csv exists:', Path('processed/train_dataset.csv').exists())

cwd: /Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project 
python: /Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project /.venv/bin/python
phase6 script exists: True
train csv exists: True


In [7]:
# Dependencies (safe to rerun)
%pip install -q numpy pandas gymnasium stable-baselines3 transformers torch scikit-learn sec-edgar-downloader

Note: you may need to restart the kernel to use updated packages.


In [9]:
# Phase 6 strict SEC ablation (FinBERT embedding mode)
import subprocess
import sys

# Provide SEC user-agent identity for EDGAR downloads.
COMPANY_NAME = 'UTAustinMSIS-PharmaTradeMM'
EMAIL = 'ajjaiswal5.imp@gmail.com'  # TODO: replace with your real email

collect_cmd = [
    sys.executable,
    'phase6_sec_collect_filings.py',
    '--project-root', str(PROJECT_DIR),
    '--feature-config-path', 'processed/feature_config.json',
    '--company-name', COMPANY_NAME,
    '--email', EMAIL,
    '--forms', '10-Q,10-K',
    '--output-root', 'sec_filings',
    '--download-root', 'sec_filings/raw_downloads',
]
print('Collecting SEC filings:', ' '.join(collect_cmd))
subprocess.run(collect_cmd, check=True)

FILINGS_META = 'sec_filings/filings_metadata.csv'   # columns: filing_id,ticker,filed_at,form_type[,text_path]
FILINGS_TEXT_ROOT = 'sec_filings/text'

build_cmd = [
    sys.executable,
    'phase6_sec_finbert_pipeline.py',
    '--project-root', str(PROJECT_DIR),
    '--feature-config-path', 'processed/feature_config.json',
    '--train-path', 'processed/train_dataset.csv',
    '--test-path', 'processed/test_dataset.csv',
    '--filings-metadata-path', FILINGS_META,
    '--filings-text-root', FILINGS_TEXT_ROOT,
    '--output-filings-emb-path', 'processed/sec_filing_embeddings.csv',
    '--output-daily-emb-path', 'processed/sec_daily_embeddings.csv',
    '--output-feature-config-path', 'processed/sec_embedding_feature_config.json',
    '--model-name', 'ProsusAI/finbert',
    '--chunk-size', '420',
    '--chunk-overlap', '80',
    '--pca-dim', '64',
]
print('Building strict SEC embeddings:', ' '.join(build_cmd))
subprocess.run(build_cmd, check=True)

train_cmd = [
    sys.executable,
    'phase6_sec_ppo.py',
    '--project-root', str(PROJECT_DIR),
    '--timesteps', '60000',
    '--search-timesteps', '15000',
    '--seeds', '7,42',
    '--val-split-date', '2021-01-01',
    '--sent-clip', '3.0',
    '--sec-clip', '5.0',
    '--sec-feature-mode', 'finbert',
    '--sec-daily-emb-path', 'processed/sec_daily_embeddings.csv',
    '--sec-feature-config-path', 'processed/sec_embedding_feature_config.json',
    '--seed', '42',
    '--window-size', '20',
    '--device', 'auto',
]
print('Running training:', ' '.join(train_cmd))
subprocess.run(train_cmd, check=True)

Project date range: 2018-02-20 to 2023-12-29
Tickers: ['PFE', 'JNJ', 'MRK', 'ABBV', 'BMY', 'AMGN', 'GILD', 'BIIB']
Forms: ['10-Q', '10-K']

Saved strict Phase 6 SEC inputs:
 - /Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project /sec_filings/filings_metadata.csv
 - /Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project /sec_filings/text
Rows: 188 | Tickers: 8 | Date range: 2018-02-21 to 2023-11-08
Building strict SEC embeddings: /Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project /.venv/bin/python phase6_sec_finbert_pipeline.py --project-root /Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project  --feature-config-path processed/feature_config.json --train-path processed/train_dataset.csv --test-path processed/test_dataset.csv --filings-metadata-path sec_filings/filings_metadata.csv --filings-text-root sec_filings/text --output-filings-emb-path processed/sec_filing_embeddings.csv --output-daily-emb-path processed/sec_daily_embeddings.csv --output-feature-config-pat

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 75672.39it/s]
BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved strict Phase 6 SEC artifacts:
 - /Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project /processed/sec_filing_embeddings.csv
 - /Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project /processed/sec_daily_embeddings.csv
 - /Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project /processed/sec_embedding_feature_config.json
SEC feature count: 66
Embedded filings: 188 | skipped: 0
Running training: /Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project /.venv/bin/python phase6_sec_ppo.py --project-root /Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project  --timesteps 60000 --search-timesteps 15000 --seeds 7,42 --val-split-date 2021-01-01 --sent-clip 3.0 --sec-clip 5.0 --sec-feature-mode finbert --sec-daily-emb-path processed/sec_daily_embeddings.csv --sec-feature-config-path processed/sec_embedding_feature_config.json --seed 42 --window-size 20 --device auto
Using PPO params from: /Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project /results/phase4_tuning/b

CompletedProcess(args=['/Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project /.venv/bin/python', 'phase6_sec_ppo.py', '--project-root', '/Users/aj5/Main drive/UT Austin MSIS/Deep Learning/Project ', '--timesteps', '60000', '--search-timesteps', '15000', '--seeds', '7,42', '--val-split-date', '2021-01-01', '--sent-clip', '3.0', '--sec-clip', '5.0', '--sec-feature-mode', 'finbert', '--sec-daily-emb-path', 'processed/sec_daily_embeddings.csv', '--sec-feature-config-path', 'processed/sec_embedding_feature_config.json', '--seed', '42', '--window-size', '20', '--device', 'auto'], returncode=0)

In [10]:
# Inspect key outputs
from pathlib import Path
import pandas as pd

base = Path('results/phase6_sec')
paths = {
    'filing_embeddings': Path('processed/sec_filing_embeddings.csv'),
    'daily_embeddings': Path('processed/sec_daily_embeddings.csv'),
    'sec_feature_cfg': Path('processed/sec_embedding_feature_config.json'),
    'metrics': base / 'phase6_sec_metrics.csv',
    'trials': base / 'phase6_robust_search_trials.csv',
    'comparison': base / 'phase3_phase4_phase5_phase6_comparison.csv',
    'interpretability': base / 'phase6_sec_interpretability.csv',
    'metadata': base / 'phase6_run_metadata.json',
}
for name, p in paths.items():
    print(f'{name}:', 'OK' if p.exists() else 'MISSING', '-', p)

if paths['metrics'].exists():
    display(pd.read_csv(paths['metrics']))
if paths['comparison'].exists():
    display(pd.read_csv(paths['comparison']))

filing_embeddings: OK - processed/sec_filing_embeddings.csv
daily_embeddings: OK - processed/sec_daily_embeddings.csv
sec_feature_cfg: OK - processed/sec_embedding_feature_config.json
metrics: OK - results/phase6_sec/phase6_sec_metrics.csv
trials: OK - results/phase6_sec/phase6_robust_search_trials.csv
comparison: OK - results/phase6_sec/phase3_phase4_phase5_phase6_comparison.csv
interpretability: OK - results/phase6_sec/phase6_sec_interpretability.csv
metadata: OK - results/phase6_sec/phase6_run_metadata.json


,strategy,cumulative_return_pct,ann_sharpe,ann_sortino,max_drawdown_pct,win_rate,alpha_vs_xph_ann_pct
0,PPO_Phase6_SEC,27.636315,0.768130,1.275613,-15.603817,0.526946,16.073845
1,BuyHold_Equal,9.625921,0.377820,0.613220,-17.925456,0.508982,7.737318
2,EqualWeight_Monthly,8.460957,0.339634,0.554942,-18.982354,0.500998,7.237284
3,XPH,-8.074730,-0.090723,-0.145941,-22.619079,0.491018,0.000000
4,Momentum_20D,-8.401114,-0.171261,-0.254539,-24.221550,0.465070,-0.998788


,metric,phase3_price_only,phase4_price_sentiment,phase5_frozen,phase6_sec,delta_p6_minus_p4,delta_p6_minus_p5
0,cumulative_return_pct,12.106765,-2.700687,25.431256,27.636315,30.337002,2.205059
1,ann_sharpe,0.445379,0.001324,0.613396,0.768130,0.766806,0.154734
2,ann_sortino,0.718082,0.002046,1.061182,1.275613,1.273568,0.214431
3,max_drawdown_pct,-15.215891,-18.387630,-17.527683,-15.603817,2.783814,1.923867
4,alpha_vs_xph_ann_pct,9.152787,2.205872,16.125121,16.073845,13.867973,-0.051276
